# 05 — Post-processing optimization

**Purpose:** compare threshold and component area using cached v2 probability maps and a fixed tuning/holdout split  
**Inputs:** v2 best checkpoint and deterministic validation images  
**Outputs:** displayed PQ/SQ/RQ tables; no weights or submission written  
**Dependencies:** project `.conda`, MAGFiLO data, trained v2 checkpoint, `src/filament_seg`.  
**Execution:** run top to bottom from the root or `notebooks/`; no training occurs.  
**Runtime:** four 2048² inference passes, then CPU-only sweeps  
**Related:** `02_training.ipynb` produces weights, `03_submission.ipynb` consumes selected settings.

PQ here is diagnostic and not verified against the organizer's exact matcher.


In [ ]:
from pathlib import Path
import sys
import numpy as np
import matplotlib.pyplot as plt
import torch
cwd = Path.cwd().resolve()
project_root = cwd if (cwd/'src/filament_seg/dataset.py').is_file() else cwd.parent
assert (project_root/'src/filament_seg/dataset.py').is_file()
assert Path(sys.executable).resolve() == (project_root/'.conda/python.exe').resolve(), sys.executable
if str(project_root/'src') not in sys.path: sys.path.insert(0,str(project_root/'src'))
from filament_seg.dataset import SolarFilamentDataset
from filament_seg.model import build_model
from filament_seg.train import load_checkpoint
from filament_seg.inference import predict_tiled, iter_probability_instances
from filament_seg.evaluate import diagnostic_panoptic_quality
from filament_seg.utils import get_device
device = get_device()
base = project_root/'data/filament-segmentation-2026/MAGFiLO_1.0_Kaggle_2026/train'
dataset = SolarFilamentDataset(base/'train_images',base/'MAGFiLO_1.0_Annotations_kaggle2026_train.json')
val_indices = np.random.default_rng(42).permutation(len(dataset))[565:].tolist()
checkpoint_path = project_root/'checkpoints/unet_resnet34_v2/best_model.pth'
if not checkpoint_path.is_file(): raise FileNotFoundError(checkpoint_path)
model = build_model(pretrained=False).to(device)
checkpoint = load_checkpoint(checkpoint_path,model,map_location=device)
assert checkpoint.get('epoch',0)>0 and not any(r.get('smoke_test') for r in checkpoint.get('history',[]))
model.eval()
print('Checkpoint epoch:',checkpoint.get('epoch'),'device:',device,'validation images:',len(val_indices))
from filament_seg.experiments import score_cached_predictions
import pandas as pd
import time

## Cache one inference per image

The small default subset is a **software and research smoke check**. For model selection, increase to a predeclared validation sample or all 142 images and keep a separate untouched holdout. Each tiling overlap requires new inference; threshold and area sweeps reuse these maps.

In [ ]:
SELECTED_INDICES = val_indices[:4]
cached = []
for idx in SELECTED_INDICES:
    sample=dataset[idx]
    started=time.perf_counter()
    probability=predict_tiled(model,sample['image'],tile_size=512,overlap=128,device=device)
    cached.append({'filename':sample['filename'],'probability_map':probability,'gt_instances':[item['mask'] for item in dataset.iter_instances(idx,0)]})
    print(sample['filename'],'inference seconds:',round(time.perf_counter()-started,2))
tuning_records, holdout_records = cached[:2], cached[2:]
assert {r['filename'] for r in tuning_records}.isdisjoint({r['filename'] for r in holdout_records})

## One-variable-at-a-time diagnostic sweep

Historical full-validation references: v1 0.275885 at 0.75/300; v2 0.386869 at 0.75/300; v2 0.400488 at 0.70/500. These numbers are not recomputed here and must not be compared directly with the four-image subset.

In [ ]:
candidates=[(0.75,300),(0.70,300),(0.75,500)]
results=[]
for threshold,area in candidates:
    started=time.perf_counter()
    row=score_cached_predictions(tuning_records,threshold=threshold,min_area=area)
    row['tuning_seconds']=time.perf_counter()-started
    results.append(row)
table=pd.DataFrame(results).sort_values('global_pq',ascending=False)
display(table[['threshold','min_area','images','global_pq','sq','rq','tp','fp','fn','tuning_seconds']])
selected=table.iloc[0]
holdout=score_cached_predictions(holdout_records,threshold=float(selected['threshold']),min_area=int(selected['min_area']))
print('Selected on tuning:',float(selected['threshold']),int(selected['min_area']))
print('Independent holdout:',holdout)
print('Subset scores are diagnostic only; no full-validation improvement is claimed.')

## Controlled next experiments

Morphological opening can erase thin filaments, and closing can merge neighbors. Test each operation against the cached maps with the same holdout; record PQ, SQ, RQ, TP/FP/FN, Dice/IoU, and runtime. Connectivity and elongation filters also need isolated sweeps. Tiling overlaps of 0/64/128/256 pixels require separate full-resolution inference caches and should be compared on identical images. The organizer's exact scorer remains unverified.